In [ ]:
import numpy as np
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import chisq, fisherz, gsq, kci, d_separation
from causallearn.graph.SHD import SHD
from causallearn.utils.DAG2CPDAG import dag2cpdag
from causallearn.utils.TXT2GeneralGraph import txt2generalgraph

data_path = "./TestData/data_linear_10.txt"
graph_path = "./TestData/graph.10.txt"
cache_file = "./citest_cache_dataname_kci.json" 

data = np.loadtxt(data_path, skiprows=1)
graph = txt2generalgraph(graph_path)
graph = dag2cpdag(graph)
num_edges = graph.get_num_edges()

In [ ]:
# Define objective function (crossvalidation and return the mean shd)
def train_pc_algorithm(config):
    model = pc(data, alpha=0.05, indep_test=config["indep_test"], cache_path=cache_file)
    shd = SHD(graph, model.G)
    print("SHD: %.2f" % shd)
    return 1 - shd.get_shd() # minimize SHD

In [ ]:
from ConfigSpace import ConfigurationSpace
from ConfigSpace.hyperparameters import UniformIntegerHyperparameter, CategoricalHyperparameter
from smac.facade.smac_bb_facade import SMAC4BB
from smac.scenario.scenario import Scenario

# Define hyperparameters
configspace = ConfigurationSpace()
indep_test = CategoricalHyperparameter("indep_test", ["fisherz", "chisq", "gsq", "kci"], default_value="fisherz")
configspace.add_hyperparameter(indep_test)

scenario = Scenario({
    "run_obj": "quality",   # optimize objective function (either quality or runtime)
    "runcount-limit": 3,    # max number of function evaluations
    "cs": configspace, 
    "deterministic": False,
})

def_value = train_pc_algorithm(configspace.get_default_configuration())
print("Default Value: %.2f" % def_value)

smac = SMAC4BB(
    scenario=scenario, 
    rng=np.random.RandomState(42),
    tae_runner=train_pc_algorithm,
)

best_found_config = smac.optimize()
best_value = train_pc_algorithm(best_found_config)
print("Optimized Value: %.2f" % best_value)